# Stage 2 Baselines: DeepLabV3+, TransUNet, Swin-UNet vs. FAB-UNet

Three architectures that all **require ImageNet/Swin-pretrained weights downloaded from the
internet** to perform fairly. Unlike Stage 1, a comparison here is only meaningful if the
pretrained weights actually loaded -- so **Section 1 below downloads and sanity-checks every
pretrained component *before* any training starts**, so a blocked download fails loudly in
seconds rather than silently producing a crippled baseline after hours of training.

**Before running this notebook:** confirm Kaggle's **Internet** toggle is ON (Settings panel,
right sidebar) -- all three architectures need it.

**Architectures covered and why each is a relevant comparison:**
- **DeepLabV3+** (Chen et al., 2018; ResNet-50 backbone) — atrous spatial pyramid pooling, a
  fundamentally different multi-scale strategy than FAB-UNet's spectral attention.
- **TransUNet** (Chen et al., 2021) — CNN encoder + Transformer bottleneck; tests whether
  *global* self-attention over the whole image improves plaque boundary detection beyond
  FAB-UNet's more local mechanisms.
- **Swin-UNet** (Cao et al., 2021) — a fully Transformer-based U-Net; the strongest "modern"
  baseline reviewers typically expect to see in a 2024+ segmentation paper.

Fill in your own FAB-UNet test-set numbers in the **reference row** cell near the end, same as
in the Stage 1 notebook.

## 0-4. Shared pipeline (identical to the FAB-UNet notebook)

In [1]:
# ------------------------------------------------------------------
# 0. Package installation (safe no-ops if already present, as on Kaggle)
# ------------------------------------------------------------------
import subprocess, sys, importlib

def _ensure(pkg, import_name=None):
    import_name = import_name or pkg
    try:
        importlib.import_module(import_name)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)

for _pkg, _imp in [("opencv-python-headless", "cv2"), ("albumentations", "albumentations"),
                    ("seaborn", "seaborn"), ("pandas", "pandas"), ("scikit-learn", "sklearn"),
                    ("scipy", "scipy"), ("tqdm", "tqdm"), ("timm", "timm")]:
    _ensure(_pkg, _imp)
print("Dependency check complete.")

Dependency check complete.


In [2]:
import os, gc, glob, random, json, math, time, warnings
from pathlib import Path
from dataclasses import dataclass, field

import numpy as np
import pandas as pd
import cv2
cv2.setNumThreads(0)   # avoid OpenCV thread explosion (RAM)
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset

import torchvision
import timm
from torchvision.models._utils import IntermediateLayerGetter

from sklearn.model_selection import KFold, StratifiedKFold, train_test_split
from scipy.ndimage import distance_transform_edt
from tqdm.auto import tqdm

try:
    import albumentations as A
    HAS_ALBUMENTATIONS = True
except ImportError:
    HAS_ALBUMENTATIONS = False

warnings.filterwarnings("ignore")
print("Imports OK. Albumentations available:", HAS_ALBUMENTATIONS)

Imports OK. Albumentations available: True


In [3]:
SEED = 42

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)
torch.backends.cudnn.benchmark = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)
if DEVICE.type == "cuda":
    print(torch.cuda.get_device_name(0))

Using device: cuda
Tesla T4


In [4]:
@dataclass
class CFG:
    DATA_ROOT: str = None
    TRAIN_IMAGE_DIR: str = None
    TRAIN_MASK_DIR: str = None
    TEST_IMAGE_DIR: str = None
    TEST_MASK_DIR: str = None

    PREP_ROOT: str = "/kaggle/working/preprocessed_384"
    OUTPUT_DIR: str = "/kaggle/working/outputs_stage2"

    IMG_SIZE: int = 384
    FORCE_REPREPROCESS: bool = False

    BATCH_SIZE: int = 6          # lower to 4 if CUDA OOM
    EPOCHS: int = 120
    LR: float = 1e-3
    K_FOLDS: int = 5
    EARLY_STOP_PATIENCE: int = 20
    NUM_WORKERS: int = 0          # 0 = no worker processes; fixes "DataLoader worker killed" (host-RAM OOM)
    MC_DROPOUT_P: float = 0.30
    MC_SAMPLES: int = 10

CFG = CFG()
CFG.DATA_ROOT = "/kaggle/input/datasets/chinnisiri/plaque-dataset/Plaque"

Path(CFG.PREP_ROOT).mkdir(parents=True, exist_ok=True)
Path(CFG.OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
print("Preprocessed data will be cached in:", CFG.PREP_ROOT)
print("All figures / models / csv reports will be saved in:", CFG.OUTPUT_DIR)

Preprocessed data will be cached in: /kaggle/working/preprocessed_384
All figures / models / csv reports will be saved in: /kaggle/working/outputs_stage2


In [5]:
import re

IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}
MASK_KEYWORDS  = ["mask", "masks", "gt", "ground_truth", "groundtruth", "label",
                  "labels", "segmentation", "annot"]
MARKED_KEYWORDS = ["marked", "annotated", "yellow"]
IMAGE_KEYWORDS = ["image", "images", "img", "raw", "original", "us_", "ultrasound"]

def _print_input_tree(base, max_depth=4):
    base = Path(base)
    print(f"Directory tree under {base} (for manual CFG.DATA_ROOT inspection):")
    for p in sorted(base.rglob("*")):
        if p.is_dir():
            depth = len(p.relative_to(base).parts)
            if depth <= max_depth:
                print("  " * depth + p.name + "/")

def _auto_detect_data_root():
    if CFG.DATA_ROOT is not None:
        return Path(CFG.DATA_ROOT)
    candidates = []
    search_base = Path("/kaggle/input")
    if search_base.exists():
        for p in search_base.rglob("*"):
            if p.is_dir():
                names = [d.name.lower() for d in p.iterdir() if d.is_dir()]
                has_train = any("train" in n for n in names)
                has_test  = any("test" in n for n in names)
                if has_train and has_test:
                    candidates.append(p)
    if candidates:
        candidates.sort(key=lambda p: len(p.parts))
        return candidates[0]
    if search_base.exists():
        _print_input_tree(search_base)
    raise FileNotFoundError(
        "Could not auto-detect a folder containing both train/ and test/ under "
        "/kaggle/input. Check the printed tree above and set CFG.DATA_ROOT manually "
        "to the dataset root, e.g. CFG.DATA_ROOT = '/kaggle/input/plaque-dataset/Plaque'."
    )

def _find_split_dir(data_root, split_name):
    for d in data_root.iterdir():
        if d.is_dir() and split_name in d.name.lower():
            return d
    raise FileNotFoundError(f"No '{split_name}' folder found under {data_root}")

def _find_role_dirs(split_dir, keywords):
    hits = []
    for d in [split_dir] + [x for x in split_dir.rglob("*") if x.is_dir()]:
        if any(k in d.name.lower() for k in keywords):
            hits.append(d)
    return hits

def _list_images(folder):
    files = [f for f in folder.rglob("*") if f.suffix.lower() in IMAGE_EXTS]
    return files

_MASK_SUFFIX_RE = re.compile(
    r"[_-](mask|masks|gt|ground[_-]?truth|label|labels|labeled|labelled|"
    r"segmentation|seg|annot|annotated|marked|binary)$"
)

def _stem_key(path):
    stem = path.stem.lower()
    m = _MASK_SUFFIX_RE.search(stem)
    if m:
        stem = stem[: m.start()]
    return stem

def discover_pairs(split_dir):
    mask_dirs = _find_role_dirs(split_dir, MASK_KEYWORDS)
    marked_dirs = _find_role_dirs(split_dir, MARKED_KEYWORDS)
    image_dirs = [d for d in _find_role_dirs(split_dir, IMAGE_KEYWORDS)
                  if d not in mask_dirs and d not in marked_dirs]
    mask_source_dirs = mask_dirs if mask_dirs else marked_dirs
    mask_is_marked = not mask_dirs and bool(marked_dirs)

    if image_dirs and mask_source_dirs and set(image_dirs) != set(mask_source_dirs):
        img_files = []
        for d in image_dirs:
            img_files += _list_images(d)
        mask_files = []
        for d in mask_source_dirs:
            mask_files += _list_images(d)
    else:
        all_files = _list_images(split_dir)
        mask_kw_all = MASK_KEYWORDS + MARKED_KEYWORDS
        mask_files = [f for f in all_files if any(k in f.stem.lower() for k in mask_kw_all)]
        mask_file_set = set(mask_files)
        img_files = [f for f in all_files if f not in mask_file_set]
        mask_is_marked = bool(mask_files) and not any(
            k in f.stem.lower() for f in mask_files for k in MASK_KEYWORDS
        )

    img_map = {_stem_key(p): p for p in img_files}
    mask_map = {_stem_key(p): p for p in mask_files}

    common = sorted(set(img_map) & set(mask_map))
    pairs = []
    for k in common:
        ip, mp = img_map[k], mask_map[k]
        if ip.resolve() == mp.resolve():
            continue
        pairs.append((ip, mp))
    return pairs, mask_is_marked

DATA_ROOT = _auto_detect_data_root()
print("Detected DATA_ROOT:", DATA_ROOT)

train_dir = Path(CFG.TRAIN_IMAGE_DIR).parent if CFG.TRAIN_IMAGE_DIR else _find_split_dir(DATA_ROOT, "train")
test_dir  = Path(CFG.TEST_IMAGE_DIR).parent  if CFG.TEST_IMAGE_DIR  else _find_split_dir(DATA_ROOT, "test")

train_pairs, train_marked = discover_pairs(train_dir)
test_pairs,  test_marked  = discover_pairs(test_dir)

print(f"Train pairs found: {len(train_pairs)}  (masks derived from 'marked' images: {train_marked})")
print(f"Test  pairs found: {len(test_pairs)}  (masks derived from 'marked' images: {test_marked})")
assert len(train_pairs) > 0 and len(test_pairs) > 0, (
    "No pairs discovered — please inspect the printed DATA_ROOT tree and set "
    "CFG.TRAIN_IMAGE_DIR / CFG.TRAIN_MASK_DIR / CFG.TEST_IMAGE_DIR / CFG.TEST_MASK_DIR manually."
)

Detected DATA_ROOT: /kaggle/input/datasets/chinnisiri/plaque-dataset/Plaque
Train pairs found: 493  (masks derived from 'marked' images: False)
Test  pairs found: 48  (masks derived from 'marked' images: False)


In [6]:
IMG_SIZE = CFG.IMG_SIZE

def resize_pad(img, size=IMG_SIZE, interp=cv2.INTER_AREA):
    h, w = img.shape[:2]
    scale = size / max(h, w)
    nh, nw = max(1, int(round(h * scale))), max(1, int(round(w * scale)))
    resized = cv2.resize(img, (nw, nh), interpolation=interp)
    pad_h, pad_w = size - nh, size - nw
    top, bottom = pad_h // 2, pad_h - pad_h // 2
    left, right = pad_w // 2, pad_w - pad_w // 2
    return cv2.copyMakeBorder(resized, top, bottom, left, right, cv2.BORDER_CONSTANT, value=0)

def preprocess_image(path):
    img = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise ValueError(f"Could not read image: {path}")
    img = resize_pad(img, IMG_SIZE, cv2.INTER_AREA)
    img = cv2.fastNlMeansDenoising(img, h=10, templateWindowSize=7, searchWindowSize=21)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    img = clahe.apply(img)
    return img

def derive_mask_from_marked(marked_bgr):
    hsv = cv2.cvtColor(marked_bgr, cv2.COLOR_BGR2HSV)
    yellow = cv2.inRange(hsv, (15, 80, 80), (35, 255, 255))
    green  = cv2.inRange(hsv, (36, 40, 40), (85, 255, 255))
    annot = cv2.bitwise_or(yellow, green)
    annot = cv2.dilate(annot, np.ones((7, 7), np.uint8), iterations=2)
    contours, _ = cv2.findContours(annot, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    mask = np.zeros(annot.shape, dtype=np.uint8)
    for c in contours:
        if cv2.contourArea(c) > 30:
            cv2.drawContours(mask, [c], -1, 255, thickness=cv2.FILLED)
    if mask.sum() == 0:
        mask = annot
    return mask

def preprocess_mask(mask_path, is_marked):
    raw = cv2.imread(str(mask_path))
    if raw is None:
        raise ValueError(f"Could not read mask: {mask_path}")
    if is_marked:
        binary = derive_mask_from_marked(raw)
    else:
        gray = cv2.cvtColor(raw, cv2.COLOR_BGR2GRAY) if raw.ndim == 3 else raw
        n_unique = len(np.unique(gray))
        if n_unique > 6:
            binary = derive_mask_from_marked(raw)
        else:
            binary = (gray > 127).astype(np.uint8) * 255
    binary = resize_pad(binary, IMG_SIZE, cv2.INTER_NEAREST)
    binary = (binary > 127).astype(np.uint8) * 255
    return binary

def run_preprocessing(pairs, split_name, is_marked, force=CFG.FORCE_REPREPROCESS):
    out_img_dir = Path(CFG.PREP_ROOT) / split_name / "images"
    out_mask_dir = Path(CFG.PREP_ROOT) / split_name / "masks"
    out_img_dir.mkdir(parents=True, exist_ok=True)
    out_mask_dir.mkdir(parents=True, exist_ok=True)

    manifest = []
    for idx, (img_path, mask_path) in enumerate(tqdm(pairs, desc=f"Preprocessing {split_name}")):
        out_name = f"{idx:04d}.png"
        out_img_path = out_img_dir / out_name
        out_mask_path = out_mask_dir / out_name
        if not force and out_img_path.exists() and out_mask_path.exists():
            manifest.append((str(img_path), str(mask_path), str(out_img_path), str(out_mask_path)))
            continue
        img_p = preprocess_image(img_path)
        mask_p = preprocess_mask(mask_path, is_marked)
        cv2.imwrite(str(out_img_path), img_p)
        cv2.imwrite(str(out_mask_path), mask_p)
        manifest.append((str(img_path), str(mask_path), str(out_img_path), str(out_mask_path)))

    df = pd.DataFrame(manifest, columns=["orig_image", "orig_mask", "prep_image", "prep_mask"])
    df.to_csv(Path(CFG.PREP_ROOT) / f"{split_name}_manifest.csv", index=False)
    return df

train_manifest = run_preprocessing(train_pairs, "train", train_marked)
test_manifest  = run_preprocessing(test_pairs, "test", test_marked)
print(f"Preprocessed {len(train_manifest)} train pairs and {len(test_manifest)} test pairs.")
print(f"Cached under: {CFG.PREP_ROOT}")

Preprocessing train:   0%|          | 0/493 [00:00<?, ?it/s]

Preprocessing test:   0%|          | 0/48 [00:00<?, ?it/s]

Preprocessed 493 train pairs and 48 test pairs.
Cached under: /kaggle/working/preprocessed_384


In [7]:
def build_train_augmentation():
    if not HAS_ALBUMENTATIONS:
        return None
    try:
        return A.Compose([
            A.HorizontalFlip(p=0.5),
            A.ShiftScaleRotate(shift_limit=0.08, scale_limit=0.15, rotate_limit=12,
                               border_mode=cv2.BORDER_CONSTANT, p=0.6),
            A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
            A.RandomGamma(gamma_limit=(80, 120), p=0.3),
            A.ElasticTransform(alpha=30, sigma=5, p=0.2),
            A.GaussNoise(p=0.2),
        ])
    except Exception as e:
        print("Falling back to the v1 augmentation pipeline:", e)
        return A.Compose([
            A.HorizontalFlip(p=0.5),
            A.Rotate(limit=15, border_mode=cv2.BORDER_CONSTANT, p=0.5),
            A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=0.4),
            A.GaussNoise(p=0.2),
        ])

def _fallback_augment(img, mask):
    if random.random() < 0.5:
        img, mask = np.fliplr(img).copy(), np.fliplr(mask).copy()
    if random.random() < 0.3:
        angle = random.uniform(-15, 15)
        h, w = img.shape
        M = cv2.getRotationMatrix2D((w / 2, h / 2), angle, 1.0)
        img = cv2.warpAffine(img, M, (w, h), borderValue=0)
        mask = cv2.warpAffine(mask, M, (w, h), borderValue=0, flags=cv2.INTER_NEAREST)
    return img, mask

TRAIN_AUG = build_train_augmentation()

def compute_boundary_dist_map(mask01):
    posmask = mask01.astype(bool)
    if posmask.any() and (~posmask).any():
        pos_dist = distance_transform_edt(posmask)
        neg_dist = distance_transform_edt(~posmask)
        dist = neg_dist * (~posmask) - (pos_dist - 1) * posmask
    else:
        dist = np.zeros_like(mask01, dtype=np.float32)
    return dist.astype(np.float32)

class PlaqueDataset(Dataset):
    def __init__(self, image_paths, mask_paths, augment=False, return_dist_map=False):
        self.image_paths = list(image_paths)
        self.mask_paths = list(mask_paths)
        self.augment = augment
        self.return_dist_map = return_dist_map

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        cv2.setNumThreads(0)
        img = cv2.imread(str(self.image_paths[idx]), cv2.IMREAD_GRAYSCALE)
        mask = cv2.imread(str(self.mask_paths[idx]), cv2.IMREAD_GRAYSCALE)
        mask01 = (mask > 127).astype(np.uint8)

        if self.augment:
            if TRAIN_AUG is not None:
                out = TRAIN_AUG(image=img, mask=mask01)
                img, mask01 = out["image"], out["mask"]
            else:
                img, mask01 = _fallback_augment(img, mask01)

        img_t = torch.from_numpy(img.astype(np.float32) / 255.0).unsqueeze(0)
        mask_t = torch.from_numpy(mask01.astype(np.float32)).unsqueeze(0)

        if self.return_dist_map:
            dist_t = torch.from_numpy(compute_boundary_dist_map(mask01)).unsqueeze(0)
            return img_t, mask_t, dist_t
        return img_t, mask_t

def paths_from_manifest(manifest_df):
    return manifest_df["prep_image"].tolist(), manifest_df["prep_mask"].tolist()

train_img_paths, train_mask_paths = paths_from_manifest(train_manifest)
test_img_paths, test_mask_paths = paths_from_manifest(test_manifest)

has_plaque = np.array([cv2.imread(p, cv2.IMREAD_GRAYSCALE).max() > 127 for p in train_mask_paths])
test_has_plaque = np.array([cv2.imread(p, cv2.IMREAD_GRAYSCALE).max() > 127 for p in test_mask_paths])
print(f"{len(train_img_paths)} training images ({has_plaque.sum()} with plaque) / "
      f"{len(test_img_paths)} test images ({test_has_plaque.sum()} with plaque) ready.")

493 training images (316 with plaque) / 48 test images (24 with plaque) ready.


## 5. Loss functions (identical weights/terms to FAB-UNet: `POS_WEIGHT`, Focal-Tversky, ramped boundary loss)

In [8]:
def dice_loss(logits, targets, eps=1.0):
    p = torch.sigmoid(logits.float()).reshape(-1)
    t = targets.float().reshape(-1)
    return 1 - (2 * (p * t).sum() + eps) / (p.sum() + t.sum() + eps)

def focal_tversky_loss(logits, targets, alpha=0.3, beta=0.7, gamma=0.75, eps=1.0):
    p = torch.sigmoid(logits.float()).reshape(-1)
    t = targets.float().reshape(-1)
    tp = (p * t).sum()
    fp = (p * (1 - t)).sum()
    fn = ((1 - p) * t).sum()
    tv = (tp + eps) / (tp + alpha * fp + beta * fn + eps)
    return (1 - tv) ** gamma

def boundary_loss(logits, dist_map):
    d = dist_map.float()
    d = d / (d.abs().amax(dim=(1, 2, 3), keepdim=True) + 1e-6)
    return (torch.sigmoid(logits.float()) * d).mean()

BND_MAX = 1.0
BND_WEIGHT = {"w": 0.0}
POS_WEIGHT = 5.0

def combined_loss(outputs, target, dist_map=None, w_bce=0.3, w_dice=0.3, w_ft=0.4):
    main = outputs["main"].float()
    target = target.float()
    pos_w = torch.tensor(POS_WEIGHT, device=main.device)
    loss = (w_bce * F.binary_cross_entropy_with_logits(main, target, pos_weight=pos_w)
            + w_dice * dice_loss(main, target)
            + w_ft * focal_tversky_loss(main, target))
    if dist_map is not None and BND_WEIGHT["w"] > 0:
        loss = loss + BND_WEIGHT["w"] * boundary_loss(main, dist_map)
    for key, w in (("aux3", 0.3), ("aux2", 0.2)):
        if key in outputs:
            aux = outputs[key].float()
            loss = loss + w * (F.binary_cross_entropy_with_logits(aux, target, pos_weight=pos_w)
                               + dice_loss(aux, target))
    return loss

## 6. Metrics (pixel-level confusion matrix + per-image Dice, incl. the paper-comparable `plaque_image_dice`)

In [9]:
class ConfusionAccumulator:
    def __init__(self):
        self.tp = self.fp = self.tn = self.fn = 0

    def update(self, pred_bin, target_bin):
        pred_bin = pred_bin.astype(bool).ravel()
        target_bin = target_bin.astype(bool).ravel()
        self.tp += int(np.sum(pred_bin & target_bin))
        self.fp += int(np.sum(pred_bin & ~target_bin))
        self.fn += int(np.sum(~pred_bin & target_bin))
        self.tn += int(np.sum(~pred_bin & ~target_bin))

    def compute(self, eps=1e-9):
        precision = self.tp / (self.tp + self.fp + eps)
        recall = self.tp / (self.tp + self.fn + eps)
        f1 = 2 * precision * recall / (precision + recall + eps)
        iou = self.tp / (self.tp + self.fp + self.fn + eps)
        dice = 2 * self.tp / (2 * self.tp + self.fp + self.fn + eps)
        accuracy = (self.tp + self.tn) / (self.tp + self.tn + self.fp + self.fn + eps)
        specificity = self.tn / (self.tn + self.fp + eps)
        return dict(precision=precision, recall=recall, f1=f1, iou=iou, dice=dice,
                    accuracy=accuracy, specificity=specificity)

    def matrix(self):
        return np.array([[self.tn, self.fp], [self.fn, self.tp]])


def image_dice(pred, gt):
    s = pred.sum() + gt.sum()
    return 1.0 if s == 0 else 2.0 * (pred & gt).sum() / s


@torch.no_grad()
def evaluate_loader(model, loader, device, threshold=0.5, mc_dropout=False):
    model.eval()
    acc = ConfusionAccumulator()
    val_loss_total, n_batches = 0.0, 0
    img_dice, img_has = [], []
    for batch in loader:
        if len(batch) == 3:
            imgs, masks, dists = batch
            dists = dists.to(device)
        else:
            imgs, masks = batch
            dists = None
        imgs, masks = imgs.to(device), masks.to(device)
        outputs = model(imgs, mc_dropout=mc_dropout)
        loss = combined_loss(outputs, masks, dists)
        val_loss_total += loss.item(); n_batches += 1
        probs = torch.sigmoid(outputs["main"].float()).cpu().numpy()
        preds = (probs > threshold)
        tgts = masks.cpu().numpy() > 0.5
        acc.update(preds, tgts)
        for p, t in zip(preds, tgts):
            img_dice.append(image_dice(p, t)); img_has.append(bool(t.any()))
    metrics = acc.compute()
    metrics["loss"] = val_loss_total / max(n_batches, 1)
    img_dice, img_has = np.array(img_dice), np.array(img_has)
    metrics["mean_image_dice"] = float(img_dice.mean())
    metrics["plaque_image_dice"] = float(img_dice[img_has].mean()) if img_has.any() else float("nan")
    return metrics, acc

## 7. Training utilities (identical `EarlyStopping`/`train_one_fold`, checkpoint-selection on `val_dice`)

In [10]:
class EarlyStopping:
    def __init__(self, patience=10, mode="min"):
        self.patience, self.mode = patience, mode
        self.best, self.count, self.should_stop = None, 0, False

    def step(self, value):
        better = (self.best is None or (self.mode == "min" and value < self.best)
                  or (self.mode == "max" and value > self.best))
        if better:
            self.best, self.count = value, 0
            return True
        self.count += 1
        if self.count >= self.patience:
            self.should_stop = True
        return False


def train_one_fold(model, train_loader, val_loader, device, epochs=CFG.EPOCHS, lr=CFG.LR,
                   patience=CFG.EARLY_STOP_PATIENCE, tag="fold", use_boundary=True, verbose=True):
    model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    warm = 3
    def lr_lambda(e):
        if e < warm:
            return (e + 1) / warm
        prog = (e - warm) / max(1, epochs - warm)
        return 0.01 + 0.99 * 0.5 * (1 + math.cos(math.pi * prog))
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))
    stopper = EarlyStopping(patience=patience, mode="max")

    history = {"train_loss": [], "val_loss": [], "val_dice": [], "val_iou": [], "val_f1": []}
    best_state, best_epoch = None, 0

    for epoch in range(1, epochs + 1):
        BND_WEIGHT["w"] = BND_MAX * min(1.0, epoch / (0.4 * epochs)) if use_boundary else 0.0
        model.train()
        running_loss, n_batches = 0.0, 0
        pbar = tqdm(train_loader, desc=f"[{tag}] epoch {epoch}/{epochs}", leave=False)
        for imgs, masks, dists in pbar:
            imgs, masks, dists = imgs.to(device), masks.to(device), dists.to(device)
            optimizer.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
                outputs = model(imgs)
            loss = combined_loss(outputs, masks, dists)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            scaler.step(optimizer)
            scaler.update()
            running_loss += loss.item(); n_batches += 1
            pbar.set_postfix(loss=running_loss / n_batches)

        train_loss = running_loss / max(n_batches, 1)
        val_metrics, _ = evaluate_loader(model, val_loader, device)
        scheduler.step()

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_metrics["loss"])
        history["val_dice"].append(val_metrics["dice"])
        history["val_iou"].append(val_metrics["iou"])
        history["val_f1"].append(val_metrics["f1"])

        if stopper.step(val_metrics["dice"]):
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            best_epoch = epoch

        if verbose and (epoch <= 3 or epoch % 5 == 0 or stopper.count == 0):
            print(f"[{tag}] epoch {epoch:03d} | lr {optimizer.param_groups[0]['lr']:.2e} | "
                  f"train_loss {train_loss:.4f} | val_loss {val_metrics['loss']:.4f} | "
                  f"val_dice {val_metrics['dice']:.4f} | val_iou {val_metrics['iou']:.4f}")

        if stopper.should_stop:
            print(f"[{tag}] Early stopping at epoch {epoch} (best val_dice={stopper.best:.4f} @ epoch {best_epoch})")
            break

    if best_state is not None:
        model.load_state_dict(best_state)
        del best_state
    del optimizer, scheduler, scaler
    gc.collect()
    if device.type == "cuda":
        torch.cuda.empty_cache()
    history["best_epoch"] = best_epoch
    BND_WEIGHT["w"] = 0.0
    return model, history

## 8. Architectures: DeepLabV3+, TransUNet, Swin-UNet (classes only -- weights verified in Section 9)

In [11]:
# ============================================================================
# Pretrained-weight verification helper — used by every Stage-2 model below.
# A random-init conv1/patch_embed has a very different weight std than an
# ImageNet/Swin-pretrained one; this is a cheap, reliable sanity check that
# catches a SILENT fallback to random weights (e.g. a blocked download) that
# would otherwise make a baseline look artificially bad for no architectural
# reason at all.
# ============================================================================
def _assert_pretrained(tensor, name, pretrained_requested):
    # NOTE: comparing weight std against a fixed threshold is NOT a reliable way to
    # detect "random vs. pretrained" -- PyTorch's default Kaiming/trunc-normal init
    # produces a similar std magnitude to most pretrained checkpoints, so that check
    # would give false confidence. The real safety net is structural: torchvision's
    # `weights=...` and timm's `pretrained=True` both raise a hard exception the
    # instant a download fails (confirmed directly: a blocked host raises
    # urllib.error.HTTPError straight out of model construction) -- they do NOT
    # silently fall back to random weights. So simply reaching this print at all,
    # with no exception raised above it, already IS the confirmation.
    std = tensor.detach().float().std().item()
    print(f"  [{name}] constructed successfully (pretrained={pretrained_requested}, "
          f"weight std={std:.4f} -- informational only, not a pass/fail signal)")
    return True


# ============================================================================
# 4. DEEPLABV3+ (Chen et al., 2018): ResNet-50 (dilated, pretrained) encoder,
#    ASPP bridge, + the "+" low-level-feature decoder torchvision's own
#    deeplabv3_resnet50 does NOT include (it ships plain DeepLabV3 only).
# ============================================================================
class DeepLabV3Plus(nn.Module):
    def __init__(self, in_ch=1, pretrained=True, mc_dropout_p=0.0):
        super().__init__()
        rn_weights = torchvision.models.ResNet50_Weights.IMAGENET1K_V1 if pretrained else None
        resnet = torchvision.models.resnet50(weights=rn_weights, replace_stride_with_dilation=[False, True, True])
        self.backbone = IntermediateLayerGetter(resnet, return_layers={"layer1": "low_level", "layer4": "out"})
        self.pretrained_ok = _assert_pretrained(resnet.conv1.weight, "DeepLabV3+ ResNet50 conv1", pretrained)

        dlv3 = torchvision.models.segmentation.deeplabv3_resnet50(weights=None, weights_backbone=None)
        self.aspp = dlv3.classifier[0]      # ASPP head structure, trained from scratch (not pretrained for VOC)

        self.low_proj = nn.Sequential(nn.Conv2d(256, 48, 1, bias=False), nn.BatchNorm2d(48), nn.ReLU(inplace=True))
        self.decoder = nn.Sequential(
            nn.Conv2d(256 + 48, 256, 3, padding=1, bias=False), nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, 3, padding=1, bias=False), nn.BatchNorm2d(256), nn.ReLU(inplace=True),
        )
        self.drop = nn.Dropout2d(mc_dropout_p) if mc_dropout_p > 0 else nn.Identity()
        self.head = nn.Conv2d(256, 1, 1)
        self.in_ch = in_ch

    def forward(self, x, mc_dropout=False):
        if self.in_ch == 1 and x.shape[1] == 1:
            x = x.repeat(1, 3, 1, 1)
        feats = self.backbone(x)
        low = self.low_proj(feats["low_level"])
        high = self.aspp(feats["out"])
        high_up = F.interpolate(high, size=low.shape[-2:], mode="bilinear", align_corners=False)
        d = self.decoder(torch.cat([high_up, low], dim=1))
        if mc_dropout: self.drop.train()
        out_small = self.head(self.drop(d))
        return {"main": F.interpolate(out_small, size=x.shape[-2:], mode="bilinear", align_corners=False)}


# ============================================================================
# 5. TRANSUNET (Chen et al., 2021): ResNet-50 stages 1-3 (pretrained) feed a
#    ViT-Base bottleneck (pretrained, timm), decoded with a CUP
#    (cascaded-upsampling) decoder with CNN skip connections.
#    No official TransUNet ships in torchvision/timm -- this assembles the
#    same encoder/bottleneck/decoder recipe from their standard building blocks.
# ============================================================================
class CUPBlock(nn.Module):
    def __init__(self, cin, cskip, cout):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(cin + cskip, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
            nn.Conv2d(cout, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
        )
    def forward(self, x, skip=None):
        x = F.interpolate(x, scale_factor=2, mode="bilinear", align_corners=False)
        if skip is not None:
            if skip.shape[-2:] != x.shape[-2:]:
                skip = F.interpolate(skip, size=x.shape[-2:], mode="bilinear", align_corners=False)
            x = torch.cat([x, skip], dim=1)
        return self.conv(x)

class TransUNet(nn.Module):
    def __init__(self, in_ch=1, img_size=384, pretrained=True, mc_dropout_p=0.0):
        super().__init__()
        rn_weights = torchvision.models.ResNet50_Weights.IMAGENET1K_V1 if pretrained else None
        resnet = torchvision.models.resnet50(weights=rn_weights)
        self.stem = nn.Sequential(resnet.conv1, resnet.bn1, resnet.relu, resnet.maxpool)
        self.layer1 = resnet.layer1; self.layer2 = resnet.layer2; self.layer3 = resnet.layer3
        self.resnet_pretrained_ok = _assert_pretrained(resnet.conv1.weight, "TransUNet ResNet50 conv1", pretrained)

        vit = timm.create_model("vit_base_patch16_224", pretrained=pretrained, num_classes=0)
        self.vit_pretrained_ok = _assert_pretrained(vit.pos_embed, "TransUNet ViT-Base pos_embed", pretrained)
        vit_dim = vit.embed_dim
        self.grid = img_size // 16 // 16
        self.patch_proj = nn.Conv2d(1024, vit_dim, kernel_size=16, stride=16)
        n_tokens = self.grid * self.grid
        self.pos_embed = nn.Parameter(self._interp_pos_embed(vit.pos_embed, n_tokens, vit_dim))
        self.vit_blocks = vit.blocks
        self.vit_norm = vit.norm
        del vit.patch_embed, vit.cls_token, vit.pos_embed, vit.head   # unused: we feed CNN features directly, no CLS token

        self.bottleneck_proj = nn.Conv2d(vit_dim, 512, 1)
        self.up3 = CUPBlock(512, 1024, 256); self.up2 = CUPBlock(256, 512, 128)
        self.up1 = CUPBlock(128, 256, 64); self.up0 = CUPBlock(64, 0, 32)
        self.drop = nn.Dropout2d(mc_dropout_p) if mc_dropout_p > 0 else nn.Identity()
        self.head = nn.Conv2d(32, 1, 1)
        self.in_ch = in_ch

    @staticmethod
    def _interp_pos_embed(pos_embed, n_tokens, dim):
        pe = pos_embed[:, 1:, :]
        n_orig = pe.shape[1]; s_orig = int(math.sqrt(n_orig)); s_new = int(math.sqrt(n_tokens))
        pe = pe.reshape(1, s_orig, s_orig, dim).permute(0, 3, 1, 2)
        pe = F.interpolate(pe, size=(s_new, s_new), mode="bicubic", align_corners=False)
        return pe.permute(0, 2, 3, 1).reshape(1, s_new * s_new, dim)

    def forward(self, x, mc_dropout=False):
        if self.in_ch == 1 and x.shape[1] == 1:
            x = x.repeat(1, 3, 1, 1)
        s0 = self.stem(x); s1 = self.layer1(s0); s2 = self.layer2(s1); s3 = self.layer3(s2)
        tok = self.patch_proj(s3).flatten(2).transpose(1, 2) + self.pos_embed
        for blk in self.vit_blocks:
            tok = blk(tok)
        tok = self.vit_norm(tok)
        B, N, C = tok.shape; g = int(math.sqrt(N))
        feat = self.bottleneck_proj(tok.transpose(1, 2).reshape(B, C, g, g))
        d = self.up3(feat, s3); d = self.up2(d, s2); d = self.up1(d, s1); d = self.up0(d, None)
        if mc_dropout: self.drop.train()
        out_small = self.head(self.drop(d))
        return {"main": F.interpolate(out_small, size=x.shape[-2:], mode="bilinear", align_corners=False)}


# ============================================================================
# 6. SWIN-UNET (Cao et al., 2021): Swin-Tiny (pretrained, timm) hierarchical
#    encoder with a symmetric upsampling decoder + skip connections from every
#    stage -- same encoder/decoder "mirrored hierarchy" spirit as the paper,
#    built from timm's ready Swin-Tiny rather than a hand-rolled patch-merging
#    / patch-expanding implementation. Encoder runs at its native 224x224;
#    input is resized down for the encoder and output resized back up to the
#    pipeline's native IMG_SIZE.
# ============================================================================
class DecBlock(nn.Module):
    def __init__(self, cin, cskip, cout):
        super().__init__()
        self.reduce = nn.Conv2d(cin, cout, 1) if cin != cout else nn.Identity()
        self.conv = nn.Sequential(
            nn.Conv2d(cout + cskip, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
            nn.Conv2d(cout, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
        )
    def forward(self, x, skip):
        x = F.interpolate(self.reduce(x), scale_factor=2, mode="bilinear", align_corners=False)
        if skip.shape[-2:] != x.shape[-2:]:
            skip = F.interpolate(skip, size=x.shape[-2:], mode="bilinear", align_corners=False)
        return self.conv(torch.cat([x, skip], dim=1))

class SwinUNet(nn.Module):
    def __init__(self, in_ch=1, encoder_res=224, pretrained=True, mc_dropout_p=0.0):
        super().__init__()
        self.encoder_res = encoder_res
        self.encoder = timm.create_model("swin_tiny_patch4_window7_224", pretrained=pretrained,
                                          features_only=True, in_chans=3)
        self.pretrained_ok = _assert_pretrained(self.encoder.patch_embed.proj.weight, "Swin-UNet Swin-Tiny patch_embed", pretrained)
        chs = self.encoder.feature_info.channels()
        self.bottleneck = nn.Sequential(nn.Conv2d(chs[3], chs[3], 3, padding=1), nn.BatchNorm2d(chs[3]), nn.ReLU(inplace=True))
        self.dec3 = DecBlock(chs[3], chs[2], chs[2])
        self.dec2 = DecBlock(chs[2], chs[1], chs[1])
        self.dec1 = DecBlock(chs[1], chs[0], chs[0])
        self.final_up = nn.Sequential(nn.ConvTranspose2d(chs[0], chs[0]//2, 2, stride=2),
                                       nn.BatchNorm2d(chs[0]//2), nn.ReLU(inplace=True))
        self.drop = nn.Dropout2d(mc_dropout_p) if mc_dropout_p > 0 else nn.Identity()
        self.head = nn.Conv2d(chs[0]//2, 1, 1)
        self.in_ch = in_ch

    def forward(self, x, mc_dropout=False):
        orig_size = x.shape[-2:]
        if self.in_ch == 1 and x.shape[1] == 1:
            x = x.repeat(1, 3, 1, 1)
        x = F.interpolate(x, size=(self.encoder_res, self.encoder_res), mode="bilinear", align_corners=False)
        f0, f1, f2, f3 = [f.permute(0, 3, 1, 2).contiguous() for f in self.encoder(x)]
        b = self.bottleneck(f3)
        d = self.dec3(b, f2); d = self.dec2(d, f1); d = self.dec1(d, f0)
        d = self.final_up(d)
        if mc_dropout: self.drop.train()
        out_small = self.head(self.drop(d))
        return {"main": F.interpolate(out_small, size=orig_size, mode="bilinear", align_corners=False)}


print("Architectures defined: DeepLabV3Plus, TransUNet, SwinUNet")
print("(pretrained-weight loading is verified per-model when instantiated below; this cell only defines the classes)")

Architectures defined: DeepLabV3Plus, TransUNet, SwinUNet
(pretrained-weight loading is verified per-model when instantiated below; this cell only defines the classes)


## 9. Pretrained-weight pre-flight check — **run this before anything else below**

Builds one instance of each architecture with `pretrained=True` and does nothing but confirm
construction succeeds (i.e. every download actually completed) and run one forward pass on a
dummy batch. If a weight download is blocked, this raises a clear `HTTPError`/`URLError` here,
in seconds -- not silently, and not after hours of training.

In [12]:
print("Pre-flight pretrained-weight check (no training yet) ...")
_dummy = torch.randn(2, 1, CFG.IMG_SIZE, CFG.IMG_SIZE)

print("\n[1/3] DeepLabV3+ ...")
_m = DeepLabV3Plus(in_ch=1, pretrained=True, mc_dropout_p=0.0)
_ = _m(_dummy)
print("  -> construction + forward pass OK (pretrained weights loaded without error)")
del _m

print("\n[2/3] TransUNet ...")
_m = TransUNet(in_ch=1, img_size=CFG.IMG_SIZE, pretrained=True, mc_dropout_p=0.0)
_ = _m(_dummy)
print("  -> construction + forward pass OK (pretrained weights loaded without error)")
del _m

print("\n[3/3] Swin-UNet ...")
_m = SwinUNet(in_ch=1, pretrained=True, mc_dropout_p=0.0)
_ = _m(_dummy)
print("  -> construction + forward pass OK (pretrained weights loaded without error)")
del _m

del _dummy
torch.cuda.empty_cache()
print("\nAll three pretrained backbones verified. Safe to proceed with full training below.")

Pre-flight pretrained-weight check (no training yet) ...

[1/3] DeepLabV3+ ...
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 128MB/s]


  [DeepLabV3+ ResNet50 conv1] constructed successfully (pretrained=True, weight std=0.1232 -- informational only, not a pass/fail signal)
  -> construction + forward pass OK (pretrained weights loaded without error)

[2/3] TransUNet ...
  [TransUNet ResNet50 conv1] constructed successfully (pretrained=True, weight std=0.1232 -- informational only, not a pass/fail signal)


model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

  [TransUNet ViT-Base pos_embed] constructed successfully (pretrained=True, weight std=0.3448 -- informational only, not a pass/fail signal)
  -> construction + forward pass OK (pretrained weights loaded without error)

[3/3] Swin-UNet ...


model.safetensors:   0%|          | 0.00/114M [00:00<?, ?B/s]

  [Swin-UNet Swin-Tiny patch_embed] constructed successfully (pretrained=True, weight std=0.0532 -- informational only, not a pass/fail signal)
  -> construction + forward pass OK (pretrained weights loaded without error)

All three pretrained backbones verified. Safe to proceed with full training below.


## 10. Train/val/test split (same seed + stratification as FAB-UNet's `final_model` split, for a fair comparison)

In [13]:
set_seed(SEED)
final_tr_idx, final_va_idx = train_test_split(np.arange(len(train_img_paths)), test_size=0.1,
                                                random_state=SEED, shuffle=True, stratify=has_plaque)

final_tr_imgs = [train_img_paths[i] for i in final_tr_idx]; final_tr_masks = [train_mask_paths[i] for i in final_tr_idx]
final_va_imgs = [train_img_paths[i] for i in final_va_idx]; final_va_masks = [train_mask_paths[i] for i in final_va_idx]

final_train_ds = PlaqueDataset(final_tr_imgs, final_tr_masks, augment=True, return_dist_map=True)
final_val_ds = PlaqueDataset(final_va_imgs, final_va_masks, augment=False, return_dist_map=True)
final_train_loader = DataLoader(final_train_ds, batch_size=CFG.BATCH_SIZE, shuffle=True,
                                  num_workers=CFG.NUM_WORKERS, drop_last=True)
final_val_loader = DataLoader(final_val_ds, batch_size=CFG.BATCH_SIZE, shuffle=False,
                                num_workers=CFG.NUM_WORKERS)

test_ds = PlaqueDataset(test_img_paths, test_mask_paths, augment=False, return_dist_map=True)
test_loader = DataLoader(test_ds, batch_size=CFG.BATCH_SIZE, shuffle=False, num_workers=CFG.NUM_WORKERS)

print(f"Split: {len(final_tr_imgs)} train / {len(final_va_imgs)} val / {len(test_img_paths)} test "
      f"(identical split seed/protocol to the FAB-UNet notebook's final_model, for a fair comparison)")

Split: 443 train / 50 val / 48 test (identical split seed/protocol to the FAB-UNet notebook's final_model, for a fair comparison)


## 11. Train and evaluate each baseline

In [14]:
RESULTS = {}         # model_name -> test metrics dict
PARAM_COUNTS = {}     # model_name -> parameter count

In [15]:
# ---------------------------------------------------------------------------
# deeplabv3plus
# ---------------------------------------------------------------------------
print("="*70); print("Training: deeplabv3plus"); print("="*70)

set_seed(SEED)
model = DeepLabV3Plus(in_ch=1, pretrained=True, mc_dropout_p=CFG.MC_DROPOUT_P)
model, history_deeplabv3plus = train_one_fold(model, final_train_loader, final_val_loader, DEVICE,
                                              epochs=CFG.EPOCHS, tag="deeplabv3plus")
torch.save(model.state_dict(), f"{CFG.OUTPUT_DIR}/deeplabv3plus_final_best.pth")

test_metrics_deeplabv3plus, test_acc_deeplabv3plus = evaluate_loader(model, test_loader, DEVICE)
print("\nTest-set metrics (deeplabv3plus):")
for k, v in test_metrics_deeplabv3plus.items():
    print(f"  {k:18s}: {v:.4f}")

RESULTS["deeplabv3plus"] = test_metrics_deeplabv3plus
n_params_deeplabv3plus = sum(p.numel() for p in model.parameters())
PARAM_COUNTS["deeplabv3plus"] = n_params_deeplabv3plus
print(f"Parameters: {n_params_deeplabv3plus/1e6:.2f} M")

del model
torch.cuda.empty_cache()
gc.collect()

Training: deeplabv3plus
  [DeepLabV3+ ResNet50 conv1] constructed successfully (pretrained=True, weight std=0.1232 -- informational only, not a pass/fail signal)


[deeplabv3plus] epoch 1/120:   0%|          | 0/73 [00:00<?, ?it/s]

[deeplabv3plus] epoch 001 | lr 6.67e-04 | train_loss 0.7190 | val_loss 0.6305 | val_dice 0.2564 | val_iou 0.1471


[deeplabv3plus] epoch 2/120:   0%|          | 0/73 [00:00<?, ?it/s]

[deeplabv3plus] epoch 002 | lr 1.00e-03 | train_loss 0.5929 | val_loss 0.5966 | val_dice 0.2777 | val_iou 0.1612


[deeplabv3plus] epoch 3/120:   0%|          | 0/73 [00:00<?, ?it/s]

... (86 outputs left out)

[deeplabv3plus] epoch 065 | lr 4.59e-04 | train_loss 0.2892 | val_loss 0.4268 | val_dice 0.5843 | val_iou 0.4128


[deeplabv3plus] epoch 66/120:   0%|          | 0/73 [00:00<?, ?it/s]

[deeplabv3plus] Early stopping at epoch 66 (best val_dice=0.6064 @ epoch 46)

Test-set metrics (deeplabv3plus):
  precision         : 0.6130
  recall            : 0.7068
  f1                : 0.6566
  iou               : 0.4887
  dice              : 0.6566
  accuracy          : 0.9937
  specificity       : 0.9962
  loss              : 0.4669
  mean_image_dice   : 0.6387
  plaque_image_dice : 0.6108
Parameters: 40.35 M


38

In [16]:
# ---------------------------------------------------------------------------
# transunet
# ---------------------------------------------------------------------------
print("="*70); print("Training: transunet"); print("="*70)

set_seed(SEED)
model = TransUNet(in_ch=1, img_size=CFG.IMG_SIZE, pretrained=True, mc_dropout_p=CFG.MC_DROPOUT_P)
model, history_transunet = train_one_fold(model, final_train_loader, final_val_loader, DEVICE,
                                              epochs=CFG.EPOCHS, tag="transunet")
torch.save(model.state_dict(), f"{CFG.OUTPUT_DIR}/transunet_final_best.pth")

test_metrics_transunet, test_acc_transunet = evaluate_loader(model, test_loader, DEVICE)
print("\nTest-set metrics (transunet):")
for k, v in test_metrics_transunet.items():
    print(f"  {k:18s}: {v:.4f}")

RESULTS["transunet"] = test_metrics_transunet
n_params_transunet = sum(p.numel() for p in model.parameters())
PARAM_COUNTS["transunet"] = n_params_transunet
print(f"Parameters: {n_params_transunet/1e6:.2f} M")

del model
torch.cuda.empty_cache()
gc.collect()

Training: transunet
  [TransUNet ResNet50 conv1] constructed successfully (pretrained=True, weight std=0.1232 -- informational only, not a pass/fail signal)
  [TransUNet ViT-Base pos_embed] constructed successfully (pretrained=True, weight std=0.3448 -- informational only, not a pass/fail signal)


[transunet] epoch 1/120:   0%|          | 0/73 [00:00<?, ?it/s]

[transunet] epoch 001 | lr 6.67e-04 | train_loss 0.8805 | val_loss 0.8506 | val_dice 0.0989 | val_iou 0.0520


[transunet] epoch 2/120:   0%|          | 0/73 [00:00<?, ?it/s]

[transunet] epoch 002 | lr 1.00e-03 | train_loss 0.8162 | val_loss 0.7938 | val_dice 0.1011 | val_iou 0.0532


[transunet] epoch 3/120:   0%|          | 0/73 [00:00<?, ?it/s]

... (52 outputs left out)

[transunet] epoch 41/120:   0%|          | 0/73 [00:00<?, ?it/s]

[transunet] epoch 42/120:   0%|          | 0/73 [00:00<?, ?it/s]

[transunet] Early stopping at epoch 42 (best val_dice=0.3210 @ epoch 22)

Test-set metrics (transunet):
  precision         : 0.3064
  recall            : 0.3367
  f1                : 0.3208
  iou               : 0.1911
  dice              : 0.3208
  accuracy          : 0.9878
  specificity       : 0.9934
  loss              : 0.6289
  mean_image_dice   : 0.2722
  plaque_image_dice : 0.2945
Parameters: 300.77 M


38

In [17]:
# ---------------------------------------------------------------------------
# swin_unet
# ---------------------------------------------------------------------------
print("="*70); print("Training: swin_unet"); print("="*70)

set_seed(SEED)
model = SwinUNet(in_ch=1, pretrained=True, mc_dropout_p=CFG.MC_DROPOUT_P)
model, history_swin_unet = train_one_fold(model, final_train_loader, final_val_loader, DEVICE,
                                              epochs=CFG.EPOCHS, tag="swin_unet")
torch.save(model.state_dict(), f"{CFG.OUTPUT_DIR}/swin_unet_final_best.pth")

test_metrics_swin_unet, test_acc_swin_unet = evaluate_loader(model, test_loader, DEVICE)
print("\nTest-set metrics (swin_unet):")
for k, v in test_metrics_swin_unet.items():
    print(f"  {k:18s}: {v:.4f}")

RESULTS["swin_unet"] = test_metrics_swin_unet
n_params_swin_unet = sum(p.numel() for p in model.parameters())
PARAM_COUNTS["swin_unet"] = n_params_swin_unet
print(f"Parameters: {n_params_swin_unet/1e6:.2f} M")

del model
torch.cuda.empty_cache()
gc.collect()

Training: swin_unet
  [Swin-UNet Swin-Tiny patch_embed] constructed successfully (pretrained=True, weight std=0.0532 -- informational only, not a pass/fail signal)


[swin_unet] epoch 1/120:   0%|          | 0/73 [00:00<?, ?it/s]

[swin_unet] epoch 001 | lr 6.67e-04 | train_loss 0.8130 | val_loss 0.8333 | val_dice 0.1009 | val_iou 0.0531


[swin_unet] epoch 2/120:   0%|          | 0/73 [00:00<?, ?it/s]

[swin_unet] epoch 002 | lr 1.00e-03 | train_loss 0.7310 | val_loss 0.6852 | val_dice 0.2978 | val_iou 0.1749


[swin_unet] epoch 3/120:   0%|          | 0/73 [00:00<?, ?it/s]

... (70 outputs left out)

[swin_unet] epoch 54/120:   0%|          | 0/73 [00:00<?, ?it/s]

[swin_unet] epoch 55/120:   0%|          | 0/73 [00:00<?, ?it/s]

[swin_unet] epoch 055 | lr 5.91e-04 | train_loss nan | val_loss nan | val_dice 0.0000 | val_iou 0.0000
[swin_unet] Early stopping at epoch 55 (best val_dice=0.4900 @ epoch 35)

Test-set metrics (swin_unet):
  precision         : 0.3877
  recall            : 0.6457
  f1                : 0.4844
  iou               : 0.3197
  dice              : 0.4844
  accuracy          : 0.9882
  specificity       : 0.9912
  loss              : 0.5326
  mean_image_dice   : 0.3073
  plaque_image_dice : 0.5313
Parameters: 38.46 M


38

## 12. Comparison table

**Fill in your own FAB-UNet test-set numbers below** (from your most recent FAB-UNet notebook
run) so the table includes it for context, without needing to retrain FAB-UNet in this
notebook. Replace every `None` with the printed value from that run's
`Test-set metrics (FAB-UNet):` block.

In [18]:
# <-- EDIT THESE: paste your own FAB-UNet (and, if you have it, baseline U-Net) test metrics here
FABUNET_REFERENCE = {
    "precision": None, "recall": None, "f1": None, "iou": None, "dice": None,
    "accuracy": None, "specificity": None, "mean_image_dice": None, "plaque_image_dice": None,
}
if all(v is not None for v in FABUNET_REFERENCE.values()):
    RESULTS["FAB-UNet (reference, from prior run)"] = FABUNET_REFERENCE
    PARAM_COUNTS["FAB-UNet (reference, from prior run)"] = float("nan")
else:
    print("FABUNET_REFERENCE not filled in -- skipping it in the comparison table. "
          "Edit the dict above with your FAB-UNet numbers to include it.")

FABUNET_REFERENCE not filled in -- skipping it in the comparison table. Edit the dict above with your FAB-UNet numbers to include it.


In [19]:
# ---------------------------------------------------------------------------
# Comparison table
# ---------------------------------------------------------------------------
rows = []
for name, m in RESULTS.items():
    rows.append({
        "model": name,
        "precision": m["precision"], "recall": m["recall"], "f1": m["f1"],
        "iou": m["iou"], "dice": m["dice"], "accuracy": m["accuracy"],
        "specificity": m["specificity"], "mean_image_dice": m["mean_image_dice"],
        "plaque_image_dice": m["plaque_image_dice"],
        "params_M": PARAM_COUNTS.get(name, float("nan")) / 1e6,
    })



comparison_df = pd.DataFrame(rows).set_index("model")
comparison_df.to_csv(f"{CFG.OUTPUT_DIR}/stage2_comparison.csv")
print(f"Saved comparison table to {CFG.OUTPUT_DIR}/stage2_comparison.csv")
print()
print("NOTE: the metric that is directly comparable to the Rulloni et al. (2026) dataset-paper")
print("baseline of 0.62 Dice is 'plaque_image_dice' (mean per-image Dice on images that")
print("actually contain a plaque) -- NOT the pooled 'dice' column, which aggregates every")
print("pixel across the whole test set (including easy plaque-free images) and is not the")
print("same metric, even though the two numbers can look superficially similar.")
print()
comparison_df.round(4)

Saved comparison table to /kaggle/working/outputs_stage2/stage2_comparison.csv

NOTE: the metric that is directly comparable to the Rulloni et al. (2026) dataset-paper
baseline of 0.62 Dice is 'plaque_image_dice' (mean per-image Dice on images that
actually contain a plaque) -- NOT the pooled 'dice' column, which aggregates every
pixel across the whole test set (including easy plaque-free images) and is not the
same metric, even though the two numbers can look superficially similar.



,precision,recall,f1,iou,dice,accuracy,specificity,mean_image_dice,plaque_image_dice,params_M
model,,,,,,,,,,
deeplabv3plus,0.6130,0.7068,0.6566,0.4887,0.6566,0.9937,0.9962,0.6387,0.6108,40.3470
transunet,0.3064,0.3367,0.3208,0.1911,0.3208,0.9878,0.9934,0.2722,0.2945,300.7698
swin_unet,0.3877,0.6457,0.4844,0.3197,0.4844,0.9882,0.9912,0.3073,0.5313,38.4631


## Conclusion

Each baseline here brings in a *different* multi-scale/global-context strategy than FAB-UNet:
ASPP's dilated-convolution pyramid (DeepLabV3+), a CNN+Transformer hybrid bottleneck
(TransUNet), and a fully hierarchical Transformer encoder (Swin-UNet). Because Section 9
verified every pretrained backbone loaded correctly before training began, a loss here is not
attributable to "the weights silently failed to download" -- any gap is attributable to the
architecture and this dataset's size (~500 images is small for these larger, transformer-heavy
models, which typically need more data to fully exploit their pretrained representations).

As in Stage 1, compare on **`plaque_image_dice`** against the Rulloni et al. (2026) dataset
paper's 0.62, not the pooled `dice` column.